# Black-Litterman Portfolio Optimization - Analysis Notebook

This notebook demonstrates how to analyze backtest results and explore different aspects of the portfolio optimization.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Custom modules
from utils import calculate_returns, calculate_sharpe_ratio
from performance_metrics import PerformanceAnalyzer

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

%load_ext autoreload
%autoreload 2

## 1. Load Backtest Results

In [ ]:
import pickle

# Load results
with open('../results/backtest_results.pkl', 'rb') as f:
    results = pickle.load(f)

print("Available strategies:")
for strategy in results.keys():
    print(f"  - {strategy}")

## 2. Performance Summary

In [ ]:
# Create performance summary
analyzer = PerformanceAnalyzer(risk_free_rate=0.04)

summary_data = []
for strategy_name, result in results.items():
    returns = result['portfolio_values']['returns']
    metrics = analyzer.calculate_metrics(returns)
    
    summary_data.append({
        'Strategy': strategy_name,
        'Total Return': f"{result['total_return']:.2%}",
        'Ann. Return': f"{metrics['annualized_return']:.2%}",
        'Ann. Vol': f"{metrics['annualized_volatility']:.2%}",
        'Sharpe': f"{metrics['sharpe_ratio']:.3f}",
        'Max DD': f"{metrics['max_drawdown']:.2%}",
        'Win Rate': f"{metrics['win_rate']:.2%}"
    })

summary_df = pd.DataFrame(summary_data)
summary_df

## 3. Cumulative Returns Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

for strategy_name, result in results.items():
    portfolio_values = result['portfolio_values']['portfolio_value']
    cumulative_returns = (portfolio_values / portfolio_values.iloc[0] - 1) * 100
    
    ax.plot(cumulative_returns.index, cumulative_returns.values, 
           label=strategy_name, linewidth=2.5, alpha=0.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return (%)', fontsize=12)
ax.set_title('Strategy Performance Comparison', fontsize=16, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Risk-Return Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for strategy_name, result in results.items():
    returns = result['portfolio_values']['returns']
    metrics = analyzer.calculate_metrics(returns)
    
    ax.scatter(metrics['annualized_volatility'] * 100,
              metrics['annualized_return'] * 100,
              s=200, alpha=0.6, label=strategy_name)
    
    ax.annotate(strategy_name,
               (metrics['annualized_volatility'] * 100,
                metrics['annualized_return'] * 100),
               xytext=(5, 5), textcoords='offset points',
               fontsize=10)

ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Risk-Return Profile', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Portfolio Weights Analysis (Black-Litterman)

In [ ]:
# Analyze BL weights
if 'black_litterman' in results and results['black_litterman']['weights'] is not None:
    weights = results['black_litterman']['weights']
    
    # Average weights
    avg_weights = weights.mean()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Average allocation
    avg_weights.plot(kind='bar', ax=ax1, color='steelblue', alpha=0.7)
    ax1.set_title('Average Portfolio Allocation', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Weight', fontsize=12)
    ax1.set_xlabel('Asset', fontsize=12)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Weight evolution (monthly)
    weights_monthly = weights.resample('M').last()
    ax2.stackplot(weights_monthly.index,
                 *[weights_monthly[col].values for col in weights_monthly.columns],
                 labels=weights_monthly.columns,
                 alpha=0.7)
    ax2.set_title('Weight Evolution Over Time', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Portfolio Weight', fontsize=12)
    ax2.set_xlabel('Date', fontsize=12)
    ax2.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=9)
    ax2.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
else:
    print("No weight data available for Black-Litterman strategy")

## 6. Rolling Performance Metrics

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

window = 60  # 60-day rolling window

for strategy_name, result in results.items():
    returns = result['portfolio_values']['returns']
    
    # Rolling Sharpe
    rolling_sharpe = (returns.rolling(window).mean() * 252 - 0.04) / \
                    (returns.rolling(window).std() * np.sqrt(252))
    
    ax1.plot(rolling_sharpe.index, rolling_sharpe.values,
            label=strategy_name, linewidth=2, alpha=0.7)
    
    # Rolling Vol
    rolling_vol = returns.rolling(window).std() * np.sqrt(252) * 100
    
    ax2.plot(rolling_vol.index, rolling_vol.values,
            label=strategy_name, linewidth=2, alpha=0.7)

ax1.set_ylabel('Rolling Sharpe Ratio', fontsize=12)
ax1.set_title(f'Rolling Performance Metrics ({window}-day window)', 
             fontsize=16, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5)

ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Annualized Volatility (%)', fontsize=12)
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Calculate correlation between strategies
returns_df = pd.DataFrame({
    strategy_name: result['portfolio_values']['returns']
    for strategy_name, result in results.items()
})

corr_matrix = returns_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
           center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Strategy Return Correlations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Export Results

In [ ]:
# Export summary to Excel
output_file = '../results/strategy_analysis.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Individual strategy returns
    for strategy_name, result in results.items():
        sheet_name = strategy_name[:31]  # Excel sheet name limit
        result['portfolio_values'].to_excel(writer, sheet_name=sheet_name)
    
    # Correlation matrix
    corr_matrix.to_excel(writer, sheet_name='Correlations')

print(f"Results exported to {output_file}")